In [ ]:
## Build the Unet for change detection
# build Unet
# understand the dimensions at each step.
# prepare dataloader.
# run the model
# build Unet++
# finish this by monday. end of monday, be done with this.

# have a working model, add it to portfolio.
# do all this by end of Monday.





# implement Unet for change detection
# build Unet model for change detection
# build dataloader  for the images
# run the model and see results

# understand concatenate
# understand dimensionality and input outputs of conv, convtranspose, maxpooling

# add weighted bce loss
# add dice loss
# add params from the unet++ model

# implement Unet++
# use sigmoid activation
# implement self supervision
# implement side extra outputs
# finish it

# make a report
# make a comparison report
# uplaod to github
# make a good page, portfolio

In [ ]:
# GETTING the dataset
# Download the RAR file
!wget -O vhrdata.rar "https://drive.usercontent.google.com/download?id=1GX656JqqOyBi_Ef0w65kDGVto-nHrNs9&export=download&confirm=t&uuid=6ad01a3d-5864-4f0c-ae7e-642a14a3d197"

!apt-get install unrar
!mkdir vhrdata2
!unrar x vhrdata.rar vhrdata/

Streaming output truncated to the last 5000 lines.
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/00998.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/00999.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01000.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01001.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01002.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01003.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01004.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01005.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01006.jpg      96%  OK 
Extracting  vhrdata/ChangeDetectionDataset/Real/subset/val/B/01007.jpg      96%  OK 
Extractin

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
from numpy.random import default_rng
import os

steps to do
1. load data into dataloader. check dimensions. see how the data is trained and tested. run a small model on the data.

2. create the model and train.

In [ ]:
# loading datasets
image_path = "vhrdata/oooo.jpg"
# Creating a custom dataset class
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, dir, transform=None):
        self.data_dir = dir
        self.images = os.listdir(dir)
        self.transform = transform

    # Defining the length of the dataset
    def __len__(self):
        return len(self.images)

    # Defining the method to get an item from the dataset
    def __getitem__(self, index):
        image_path = os.path.join(self.data_dir, self.images[index])
        image = np.array(Image.open(image_path))

        # Applying the transform
        if self.transform:
            image = self.transform(image)

        return image




In [ ]:
arr1 = torch.from_numpy(default_rng(41).random((2, 10, 10,  1)))
arr2 = torch.from_numpy(default_rng(43).random((2, 10, 10,  1)))

print(weighted_bce_dice_loss(arr1, arr2))

tensor(0.2714, dtype=torch.float64)


In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
from numpy.random import default_rng

def dice_coef_loss(y_true, y_pred, smooth = 1, weight = 0.5):

  y_true = y_true[:, :, :, -1]   # if it has 5 dimensions with the last dimension being 1. this basically eliminates the last dimension
  y_pred = y_pred[:, :, :, -1]

  intersection = torch.sum(y_true * y_pred)
  union = torch.sum(y_true) + weight * torch.sum(y_pred)

  return (1 - (2.0 * (intersection))/(union + smooth))


def weighted_bce_loss(y_true, y_pred):

  y_true = y_true[:, :, :, -1]   # if it has 5 dimensions with the last dimension being 1. this basically eliminates the last dimension
  y_pred = y_pred[:, :, :, -1]
  class_logloss = torch.mean(F.binary_cross_entropy( y_pred, y_true, reduction='none'), (0,1,2))
  class_weights = [0.1 * 0.9]
  weighted_bce_dice_loss = torch.sum(class_logloss * torch.tensor(class_weights))

  return weighted_bce_dice_loss


def weighted_bce_dice_loss(y_true, y_pred):
  return weighted_bce_loss(y_true, y_pred) +  torch.tensor(0.5) * dice_coef_loss(y_true, y_pred)

In [ ]:
# make the standard unit layers

nb_filter = 10
padding = 1 # Assuming padding is 1 for kernel size 3 to maintain spatial dimensions
convlayer1 = nn.Conv2d(in_channels = 3, out_channels = nb_filter , kernel_size = 3, stride = 1, padding = padding)
dropout_layer = nn.Dropout(0.2) # Example dropout rate of 0.2
batch_normalization_layer = nn.BatchNorm2d(nb_filter) # Batch norm is applied to the output channels of the previous layer
convlayer2 = nn.Conv2d(in_channels = nb_filter, out_channels = nb_filter , kernel_size = 3, stride = 1, padding = padding)
dropout_layer2 = nn.Dropout(0.2) # Example dropout rate of 0.2
batch_norm_layer2 = nn.BatchNorm2d(nb_filter)

# Example of how to use these layers in a sequence:
# x = convlayer1(input_tensor)
# x = dropout_layer(x)
# x = batch_normalization_layer(x)
# x = convlayer2(x)
# x = dropout_layer2(x)
# x = batch_norm_layer2(x)

In [ ]:
import os
import numpy as np
from keras import Input, Model
from keras.layers import Dense, Conv2D, MaxPooling2D, Dropout, Flatten, merge, Concatenate,Conv2DTranspose,add,Concatenate,Add,Subtract
from keras.optimizers import Adam,SGD
from keras.utils import plot_model
from keras import backend as K
from keras.layers import add,BatchNormalization,UpSampling2D
from keras.layers import Embedding,Input,Conv2D,Conv3D,Lambda,concatenate,Flatten,Dense,Dropout,MaxPooling2D,Activation,GlobalAveragePooling2D,GlobalAveragePooling3D,BatchNormalization
from keras import regularizers
import tensorflow as tf
from keras import objectives
from keras.regularizers import l2
from keras.callbacks import Callback

SEED = 1998
np.random.seed(SEED)
session_conf = tf.ConfigProto(intra_op_parallelism_threads=1,
                              inter_op_parallelism_threads=1)
tf.set_random_seed(SEED)
sess = tf.Session(graph=tf.get_default_graph(), config=session_conf)
K.set_session(sess)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


# def dice_coef(y_true, y_pred, smooth=1, weight=0.5):
#     """
#     加权后的dice coefficient
#     """
#     y_true = y_true[:, :, :, -1]  # y_true[:, :, :, :-1]=y_true[:, :, :, -1] if dim(3)=1 等效于[8,256,256,1]==>[8,256,256]
#     y_pred = y_pred[:, :, :, -1]
#     intersection = K.sum(y_true * y_pred)
#     union = K.sum(y_true) + weight * K.sum(y_pred)
#     # K.mean((2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth))
#     return ((2. * intersection + smooth) / (union + smooth))  # not working better using mean
# def dice_coef_loss(y_true, y_pred):
#     """
#     目标函数
#     """
#     return 1 - dice_coef(y_true, y_pred)
# def weighted_bce_dice_loss(y_true,y_pred):
#     class_loglosses = K.mean(K.binary_crossentropy(y_true, y_pred), axis=[0, 1, 2])

#     class_weights = [0.1, 0.9]#note that the weights can be computed automatically using the training smaples
#     weighted_bce = K.sum(class_loglosses * K.constant(class_weights))

#     # return K.weighted_binary_crossentropy(y_true, y_pred,pos_weight) + 0.35 * (self.dice_coef_loss(y_true, y_pred)) #not work
#     return weighted_bce + 0.5 * (dice_coef_loss(y_true, y_pred))
def standard_unit(input_tensor, stage, nb_filter, kernel_size=3, mode='None'):
    x = Conv2D(nb_filter, (kernel_size, kernel_size), activation='selu', name='conv' + stage + '_1',
               kernel_initializer='he_normal', padding='same', kernel_regularizer=l2(1e-4))(input_tensor)
    x0 = x
    # x = Dropout(0.2, name='dp' + stage + '_1')(x)
    x = BatchNormalization(name='bn' + stage + '_1')(x)  # much better than dropout
    x = Conv2D(nb_filter, (kernel_size, kernel_size), activation='selu', name='conv' + stage + '_2',
               kernel_initializer='he_normal', padding='same', kernel_regularizer=l2(1e-4))(x)
    # x = Dropout(0.2, name='dp' + stage + '_2')(x)
    x = BatchNormalization(name='bn' + stage + '_2')(x)
    if mode == 'residual':
        # x=Add(name='resi'+stage)([x,input_tensor])# 维度不相同！
        x = Add(name='resi' + stage)([x, x0])

    return x

def Nest_Net2(input_shape, num_class=1, deep_supervision=False):
    nb_filter = [32, 64, 128, 256, 512]
    # nb_filter = [16, 32, 64, 128, 256]
    mode = 'residual'  # mode='residual' seems to improve better than DS
    # Handle Dimension Ordering for different backends
    bn_axis = 3
    inputs = Input(shape=input_shape)
    conv1_1 = standard_unit(inputs, stage='11', nb_filter=nb_filter[0])  # add 要求输入输出维度相同
    pool1 = MaxPooling2D((2, 2), strides=(2, 2), name='pool1')(conv1_1)  # (?,128,128,32)

    conv2_1 = standard_unit(pool1, stage='21', nb_filter=nb_filter[1], mode=mode)
    pool2 = MaxPooling2D((2, 2), strides=(2, 2), name='pool2')(conv2_1)  # (?,64,64,64)

    up1_2 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up12', padding='same')(conv2_1)
    conv1_2 = concatenate([up1_2, conv1_1], name='merge12', axis=bn_axis)  # (?,256,256,64)
    conv1_2 = standard_unit(conv1_2, stage='12', nb_filter=nb_filter[0], mode=mode)  # (?,256,256,32)

    conv3_1 = standard_unit(pool2, stage='31', nb_filter=nb_filter[2], mode=mode)
    pool3 = MaxPooling2D((2, 2), strides=(2, 2), name='pool3')(conv3_1)

    up2_2 = Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), name='up22', padding='same')(conv3_1)
    conv2_2 = concatenate([up2_2, conv2_1], name='merge22', axis=bn_axis)
    conv2_2 = standard_unit(conv2_2, stage='22', nb_filter=nb_filter[1], mode=mode)

    up1_3 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up13', padding='same')(conv2_2)
    conv1_3 = concatenate([up1_3, conv1_1, conv1_2], name='merge13', axis=bn_axis)
    conv1_3 = standard_unit(conv1_3, stage='13', nb_filter=nb_filter[0], mode=mode)  # (?,256,256,32)

    conv4_1 = standard_unit(pool3, stage='41', nb_filter=nb_filter[3], mode=mode)
    pool4 = MaxPooling2D((2, 2), strides=(2, 2), name='pool4')(conv4_1)

    up3_2 = Conv2DTranspose(nb_filter[2], (2, 2), strides=(2, 2), name='up32', padding='same')(conv4_1)
    conv3_2 = concatenate([up3_2, conv3_1], name='merge32', axis=bn_axis)
    conv3_2 =standard_unit(conv3_2, stage='32', nb_filter=nb_filter[2], mode=mode)

    up2_3 = Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), name='up23', padding='same')(conv3_2)
    conv2_3 = concatenate([up2_3, conv2_1, conv2_2], name='merge23', axis=bn_axis)
    conv2_3 = standard_unit(conv2_3, stage='23', nb_filter=nb_filter[1], mode=mode)

    up1_4 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up14', padding='same')(conv2_3)
    conv1_4 = concatenate([up1_4, conv1_1, conv1_2, conv1_3], name='merge14', axis=bn_axis)
    conv1_4 =standard_unit(conv1_4, stage='14', nb_filter=nb_filter[0], mode=mode)

    conv5_1 = standard_unit(pool4, stage='51', nb_filter=nb_filter[4], mode=mode)

    up4_2 = Conv2DTranspose(nb_filter[3], (2, 2), strides=(2, 2), name='up42', padding='same')(conv5_1)
    conv4_2 = concatenate([up4_2, conv4_1], name='merge42', axis=bn_axis)
    conv4_2 = standard_unit(conv4_2, stage='42', nb_filter=nb_filter[3], mode=mode)

    up3_3 = Conv2DTranspose(nb_filter[2], (2, 2), strides=(2, 2), name='up33', padding='same')(conv4_2)
    conv3_3 = concatenate([up3_3, conv3_1, conv3_2], name='merge33', axis=bn_axis)
    conv3_3 = standard_unit(conv3_3, stage='33', nb_filter=nb_filter[2], mode=mode)

    up2_4 = Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), name='up24', padding='same')(conv3_3)
    conv2_4 = concatenate([up2_4, conv2_1, conv2_2, conv2_3], name='merge24', axis=bn_axis)
    conv2_4 = standard_unit(conv2_4, stage='24', nb_filter=nb_filter[1], mode=mode)

    up1_5 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up15', padding='same')(conv2_4)
    conv1_5 = concatenate([up1_5, conv1_1, conv1_2, conv1_3, conv1_4], name='merge15', axis=bn_axis)
    conv1_5 =standard_unit(conv1_5, stage='15', nb_filter=nb_filter[0], mode=mode)

    nestnet_output_1 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output_1',
                              kernel_initializer='he_normal', padding='same', kernel_regularizer=l2(1e-4))(conv1_2)
    nestnet_output_2 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output_2',
                              kernel_initializer='he_normal', padding='same', kernel_regularizer=l2(1e-4))(conv1_3)
    nestnet_output_3 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output_3',
                              kernel_initializer='he_normal', padding='same', kernel_regularizer=l2(1e-4))(conv1_4)
    nestnet_output_4 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output_4',
                              kernel_initializer='he_normal', padding='same', kernel_regularizer=l2(1e-4))(conv1_5)
    # using combined loss
    conv_fuse = concatenate([conv1_2, conv1_3, conv1_4, conv1_5], name='merge_fuse', axis=bn_axis)
    nestnet_output_5 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output_5',
                              kernel_initializer='he_normal', padding='same', kernel_regularizer=l2(1e-4))(conv_fuse)


    if deep_supervision:
        model = Model(input=inputs, output=[nestnet_output_1,
                                            nestnet_output_2,
                                            nestnet_output_3,
                                            nestnet_output_4, nestnet_output_5])
        model.compile(optimizer=Adam(lr=1e-4),
                      #loss=['binary_crossentropy','binary_crossentropy','binary_crossentropy','binary_crossentropy'],
                      loss=[weighted_bce_dice_loss, weighted_bce_dice_loss, weighted_bce_dice_loss,
                            weighted_bce_dice_loss, weighted_bce_dice_loss],
                      loss_weights=[0.5, 0.5, 0.75, 0.5, 1.0],
                      metrics=['accuracy']
                      )
    else:
        model = Model(input=inputs, output=[nestnet_output_4])
        model.compile(optimizer=Adam(lr=1e-4), loss=weighted_bce_dice_loss,
                      metrics=['accuracy'])
    model.summary()
    return model

if __name__ == '__main__':
    input_shape = [256, 256, 6]
    model=Nest_Net2(input_shape,deep_supervision=True)
    output_layer=model.get_layer('output_5')
    print("the output shape is:")
    print(output_layer.output_shape)

# Task
Convert the provided Keras U-Net++ model code, including the `standard_unit` function and the main model architecture with deep supervision, into a PyTorch `nn.Module`. The conversion should accurately reflect the layers, connections, and the forward pass of the original Keras model.

## Translate `standard unit` function

### Subtask:
Convert the Keras `standard_unit` function, which includes `Conv2D`, `BatchNormalization`, and `Add` (for residual mode) layers, into a PyTorch `nn.Module`.


**Reasoning**:
Convert the Keras `standard_unit` function to a PyTorch `nn.Module` named `StandardUnit`.



In [ ]:
from PIL import Image

class StandardUnit(nn.Module):
    def __init__(self, in_channels, nb_filter, kernel_size=3, mode='None'):
        super(StandardUnit, self).__init__()
        self.mode = mode
        padding = kernel_size // 2 # Calculate padding to maintain spatial dimensions

        self.conv1 = nn.Conv2d(in_channels, nb_filter, kernel_size=kernel_size, stride=1, padding=padding)
        self.bn1 = nn.BatchNorm2d(nb_filter)

        self.conv2 = nn.Conv2d(nb_filter, nb_filter, kernel_size=kernel_size, stride=1, padding=padding)
        self.bn2 = nn.BatchNorm2d(nb_filter)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = F.selu(out)
        out = self.bn1(out)

        out = self.conv2(out)
        out = F.selu(out)
        out = self.bn2(out)

        if self.mode == 'residual':
            # Assuming the input and output dimensions match for residual connection
            # If dimensions don't match, a 1x1 convolution or other method would be needed
            out += identity

        return out

**Reasoning**:
The previous command failed because `nn` was not defined. Add the necessary import from `torch`.



In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from PIL import Image

class StandardUnit(nn.Module):
    def __init__(self, in_channels, nb_filter, kernel_size=3, mode='None'):
        super(StandardUnit, self).__init__()
        self.mode = mode
        padding = kernel_size // 2 # Calculate padding to maintain spatial dimensions
        self.nb_filter = nb_filter

        self.conv1 = nn.Conv2d(in_channels, nb_filter, kernel_size=kernel_size, stride=1, padding=padding)
        self.bn1 = nn.BatchNorm2d(nb_filter)

        self.conv2 = nn.Conv2d(nb_filter, nb_filter, kernel_size=kernel_size, stride=1, padding=padding)
        self.bn2 = nn.BatchNorm2d(nb_filter)

        # Residual connection handling
        if mode == 'residual' and in_channels != nb_filter:
            self.residual_conv = nn.Conv2d(in_channels, nb_filter, kernel_size=1, stride=1, padding=0)
        else:
            self.residual_conv = None


    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = F.selu(out)
        out = self.bn1(out)

        out = self.conv2(out)
        out = F.selu(out)
        out = self.bn2(out)

        if self.mode == 'residual':
            if self.residual_conv:
                identity = self.residual_conv(identity)
            out += identity

        return out

## Build the pytorch u-net++ model structure

### Subtask:
Create a main PyTorch `nn.Module` class that replicates the U-Net++ architecture from the Keras code, including the encoder, decoder, nested and dense skip connections using PyTorch layers and operations (like `torch.nn.Conv2d`, `torch.nn.MaxPool2d`, `torch.nn.ConvTranspose2d`, `torch.cat`).


**Reasoning**:
Define the PyTorch U-Net++ model based on the Keras implementation, incorporating the StandardUnit module, pooling, upsampling, and skip connections as described in the instructions.



In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F # Import F for activation functions

class UNetPlusPlus(nn.Module):
    def __init__(self, input_channels, num_classes=1, deep_supervision=False):
        super(UNetPlusPlus, self).__init__()
        self.deep_supervision = deep_supervision
        nb_filter = [32, 64, 128, 256, 512]
        mode = 'residual'

        # Encoder path
        self.conv1_1 = StandardUnit(input_channels, nb_filter[0], mode=mode)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2_1 = StandardUnit(nb_filter[0], nb_filter[1], mode=mode)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3_1 = StandardUnit(nb_filter[1], nb_filter[2], mode=mode)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4_1 = StandardUnit(nb_filter[2], nb_filter[3], mode=mode)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv5_1 = StandardUnit(nb_filter[3], nb_filter[4], mode=mode)

        # Decoder path
        self.up4_2 = nn.ConvTranspose2d(nb_filter[4], nb_filter[3], kernel_size=2, stride=2)
        self.conv4_2 = StandardUnit(nb_filter[3] + nb_filter[3], nb_filter[3], mode=mode) # Concatenation adds channels

        self.up3_3 = nn.ConvTranspose2d(nb_filter[3], nb_filter[2], kernel_size=2, stride=2)
        self.conv3_3 = StandardUnit(nb_filter[2] + nb_filter[2] + nb_filter[2], nb_filter[2], mode=mode) # Concatenation adds channels

        self.up2_4 = nn.ConvTranspose2d(nb_filter[2], nb_filter[1], kernel_size=2, stride=2)
        self.conv2_4 = StandardUnit(nb_filter[1] + nb_filter[1] + nb_filter[1] + nb_filter[1], nb_filter[1], mode=mode) # Concatenation adds channels

        self.up1_5 = nn.ConvTranspose2d(nb_filter[1], nb_filter[0], kernel_size=2, stride=2)
        self.conv1_5 = StandardUnit(nb_filter[0] + nb_filter[0] + nb_filter[0] + nb_filter[0] + nb_filter[0], nb_filter[0], mode=mode) # Concatenation adds channels

        # Nested and dense skip connections
        self.up1_2 = nn.ConvTranspose2d(nb_filter[1], nb_filter[0], kernel_size=2, stride=2)
        self.conv1_2 = StandardUnit(nb_filter[0] + nb_filter[0], nb_filter[0], mode=mode)

        self.up2_2 = nn.ConvTranspose2d(nb_filter[2], nb_filter[1], kernel_size=2, stride=2)
        self.conv2_2 = StandardUnit(nb_filter[1] + nb_filter[1], nb_filter[1], mode=mode)

        self.up1_3 = nn.ConvTranspose2d(nb_filter[1], nb_filter[0], kernel_size=2, stride=2)
        self.conv1_3 = StandardUnit(nb_filter[0] + nb_filter[0] + nb_filter[0], nb_filter[0], mode=mode)

        self.up3_2 = nn.ConvTranspose2d(nb_filter[3], nb_filter[2], kernel_size=2, stride=2)
        self.conv3_2 = StandardUnit(nb_filter[2] + nb_filter[2], nb_filter[2], mode=mode)

        self.up2_3 = nn.ConvTranspose2d(nb_filter[2], nb_filter[1], kernel_size=2, stride=2)
        self.conv2_3 = StandardUnit(nb_filter[1] + nb_filter[1] + nb_filter[1], nb_filter[1], mode=mode)

        self.up1_4 = nn.ConvTranspose2d(nb_filter[1], nb_filter[0], kernel_size=2, stride=2)
        self.conv1_4 = StandardUnit(nb_filter[0] + nb_filter[0] + nb_filter[0] + nb_filter[0], nb_filter[0], mode=mode)


        # Output layers for deep supervision and final output
        self.output_1 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)
        self.output_2 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)
        self.output_3 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)
        self.output_4 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)
        self.output_5 = nn.Conv2d(nb_filter[0] * 4, num_classes, kernel_size=1) # Concatenation of conv1_2, conv1_3, conv1_4, conv1_5


    def forward(self, x):
        # Encoder path
        conv1_1 = self.conv1_1(x)
        pool1 = self.pool1(conv1_1)

        conv2_1 = self.conv2_1(pool1)
        pool2 = self.pool2(conv2_1)

        up1_2 = self.up1_2(conv2_1)
        conv1_2 = self.conv1_2(torch.cat([up1_2, conv1_1], dim=1))

        conv3_1 = self.conv3_1(pool2)
        pool3 = self.pool3(conv3_1)

        up2_2 = self.up2_2(conv3_1)
        conv2_2 = self.conv2_2(torch.cat([up2_2, conv2_1], dim=1))

        up1_3 = self.up1_3(conv2_2)
        conv1_3 = self.conv1_3(torch.cat([up1_3, conv1_1, conv1_2], dim=1))

        conv4_1 = self.conv4_1(pool3)
        pool4 = self.pool4(conv4_1)

        up3_2 = self.up3_2(conv4_1)
        conv3_2 = self.conv3_2(torch.cat([up3_2, conv3_1], dim=1))

        up2_3 = self.up2_3(conv3_2)
        conv2_3 = self.conv2_3(torch.cat([up2_3, conv2_1, conv2_2], dim=1))

        up1_4 = self.up1_4(conv2_3)
        conv1_4 = self.conv1_4(torch.cat([up1_4, conv1_1, conv1_2, conv1_3], dim=1))

        conv5_1 = self.conv5_1(pool4)

        up4_2 = self.up4_2(conv5_1)
        conv4_2 = self.conv4_2(torch.cat([up4_2, conv4_1], dim=1))

        up3_3 = self.up3_3(conv4_2)
        conv3_3 = self.conv3_3(torch.cat([up3_3, conv3_1, conv3_2], dim=1))

        up2_4 = self.up2_4(conv3_3)
        conv2_4 = self.conv2_4(torch.cat([up2_4, conv2_1, conv2_2, conv2_3], dim=1))

        up1_5 = self.up1_5(conv2_4)
        conv1_5 = self.conv1_5(torch.cat([up1_5, conv1_1, conv1_2, conv1_3, conv1_4], dim=1))

        # Output layers
        output1 = torch.sigmoid(self.output_1(conv1_2))
        output2 = torch.sigmoid(self.output_2(conv1_3))
        output3 = torch.sigmoid(self.output_3(conv1_4))
        output4 = torch.sigmoid(self.output_4(conv1_5))

        if self.deep_supervision:
            return [output1, output2, output3, output4]
        else:
            # Combined output for deep supervision
            conv_fuse = torch.cat([conv1_2, conv1_3, conv1_4, conv1_5], dim=1)
            output5 = torch.sigmoid(self.output_5(conv_fuse))
            return output5

In [ ]:
# Instantiate the model
input_channels = 6 # As determined from the Keras model's input shape
num_classes = 1 # For binary change detection
deep_supervision = True # Set to True to get multiple outputs

model = UNetPlusPlus(input_channels, num_classes, deep_supervision)

# Create dummy input data
# Assuming input images are 256x256
dummy_input = torch.randn(1, input_channels, 256, 256) # Batch size of 1

# Perform a forward pass
with torch.no_grad(): # No need to calculate gradients for this verification
    outputs = model(dummy_input)

# Print the shapes of the outputs to verify
if deep_supervision:
    print("Output shapes with deep supervision:")
    for i, output in enumerate(outputs):
        print(f"Output {i+1} shape: {output.shape}")
else:
    print(f"Output shape without deep supervision: {outputs.shape}")

RuntimeError: The size of tensor a (32) must match the size of tensor b (6) at non-singleton dimension 1